# AbirQu — Quick Start
#
# This notebook walks through the core features of AbirQu:
# installation, basic circuits, chemistry, error correction, and
# quantum communication.  Run each cell to get started!

## 1. Installation

```bash
pip install abirqu
```


In [ ]:
pip install -q abirqu

## 2. Basic Quantum Circuit

Create a Bell state, apply rotations, and measure.


In [ ]:
from abirqu.circuit import Circuit
import numpy as np

circuit = Circuit(2, name="Bell state")
circuit.h(0)
circuit.cnot(0, 1)
circuit.measure_all()

result = circuit.run(shots=1024)
print("Counts:", result["counts"])
print("Probabilities:", result["probabilities"])


## 3. Quantum Chemistry (VQE)

Minimise the energy of a simple Hamiltonian.


In [ ]:
from abirqu.circuit import Circuit
from abirqu.autodiff import parameter_shift_gradient
import numpy as np

# Simple 2-qubit VQE ansatz
def vqe_circuit(theta):
    c = Circuit(2)
    c.ry(0, theta[0])
    c.ry(1, theta[1])
    c.cnot(0, 1)
    return c

# Hamiltonian: Z⊗I + I⊗Z + 0.5*X⊗X
H = np.diag([1, -1, -1, 1]).astype(complex)
H += 0.5 * np.array([[0,0,0,1],[0,0,1,0],[0,1,0,0],[1,0,0,0]], dtype=complex)

# Initial parameters
theta = np.array([0.5, 0.3])
grad = parameter_shift_gradient(vqe_circuit(theta), theta, H)
print("Gradients:", grad.gradients)
print("Method:", grad.method, "| Evals:", grad.circuit_evals)


## 4. Quantum Error Correction

Protect a logical qubit using a repetition code.


In [ ]:
from abirqu.circuit import Circuit

# 3-qubit repetition code
code = Circuit(3)
code.h(0)  # encode
code.cnot(0, 1)
code.cnot(0, 2)

# Introduce a bit-flip error on qubit 1
code.x(1)

# Syndrome measurement
code.cnot(0, 1)
code.cnot(0, 2)

code.measure_all()
result = code.run(shots=1024)
print("Syndrome counts:", result["counts"])


## 5. Quantum Communication

BB84-style key distribution simulation.


In [ ]:
from abirqu.circuit import Circuit
import numpy as np

def bb84_round():
    n = 4
    circ = Circuit(n)
    # Alice's random bits and bases
    bits = np.random.randint(0, 2, n)
    bases = np.random.randint(0, 2, n)
    for i in range(n):
        if bits[i]:
            circ.x(i)
        if bases[i]:
            circ.h(i)
    # Bob's random bases
    bob_bases = np.random.randint(0, 2, n)
    for i in range(n):
        if bob_bases[i]:
            circ.h(i)
    circ.measure_all()
    return circ, bits, bases, bob_bases

circ, bits, a_bases, b_bases = bb84_round()
result = circ.run(shots=1)
print("Alice bits:", bits)
print("Alice bases:", a_bases)
print("Bob bases:  ", b_bases)
